# Notebook 02 – Entraînement du modèle audio (CNN)

In [13]:
import tensorflow as tf
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ===================== CONFIG =====================
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA_AUDIO = os.path.join(PROJECT_ROOT, "data", "audio")
MODEL_PATH = os.path.join(PROJECT_ROOT, "models", "audio_model.keras")

print("📁 DATA_AUDIO :", DATA_AUDIO)

# ===================== CHARGEMENT DATA =====================
X, y = [], []
classes = ['normal', 'cry', 'fight']

for label, cls in enumerate(classes):
    path = os.path.join(DATA_AUDIO, cls)
    
    if not os.path.exists(path):
        print(f"❌ Dossier manquant : {path}")
        continue

    files = [f for f in os.listdir(path) if f.endswith('.npy')]
    print(f"📂 {cls} : {len(files)} fichiers")

    for f in files:
        try:
            file_path = os.path.join(path, f)
            X.append(np.load(file_path))
            y.append(label)
        except Exception as e:
            print(f"❌ Erreur fichier {f} : {e}")

X = np.array(X)
y = np.array(y)

print(f"\n✅ Total samples : {len(X)}")

assert len(X) > 0, "❌ Dataset vide → vérifier Notebook 01"

# ===================== NORMALISATION =====================
X = X / np.max(np.abs(X))

# ===================== SPLIT =====================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train : {len(X_train)} | Test : {len(X_test)}")

# ===================== CLASS WEIGHT =====================
weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)

class_weight = {i: weights[i] for i in range(len(weights))}
print("⚖️ Class weights :", class_weight)

# ===================== AUGMENTATION =====================
def add_noise(x):
    noise = np.random.randn(*x.shape) * 0.01
    return x + noise

def augment(X, y):
    X_aug, y_aug = [], []
    
    for i in range(len(X)):
        X_aug.append(X[i])
        y_aug.append(y[i])

        X_aug.append(add_noise(X[i]))
        y_aug.append(y[i])

    return np.array(X_aug), np.array(y_aug)

X_train, y_train = augment(X_train, y_train)
print("Après augmentation :", len(X_train))

# ===================== MODELE CNN =====================
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 124, 1)),

    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ===================== CALLBACKS =====================
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        save_best_only=True
    )
]

# ===================== TRAIN =====================
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test),
    class_weight=class_weight,
    callbacks=callbacks
)

# ===================== EVALUATION =====================
y_pred = np.argmax(model.predict(X_test), axis=1)

print("\n📊 Classification Report")
print(classification_report(y_test, y_pred, target_names=classes))

# ===================== MATRICE CONFUSION =====================
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=classes,
    yticklabels=classes
)
plt.title("Matrice de confusion")
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.show()

# ===================== COURBES =====================
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title("Loss")
plt.legend()

plt.show()

print(f"\n💾 Modèle sauvegardé : {MODEL_PATH}")

# ===================== EXPORT TFLITE =====================
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

tflite_path = MODEL_PATH.replace(".keras", ".tflite")

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"📦 Modèle TFLite : {tflite_path}")

📁 DATA_AUDIO : C:\Users\AHMED MOHATI\Documents\School\S2\Programmation Mobile\Projet\aman_ai_mvp\data\audio
📂 normal : 0 fichiers
📂 cry : 0 fichiers
📂 fight : 0 fichiers

✅ Total samples : 0


AssertionError: ❌ Dataset vide → vérifier Notebook 01